In [ ]:
!pip install transformers datasets sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 15.3 MB/s eta 0:00:00


In [ ]:
import os
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from torch.utils.data import Dataset

os.environ["WANDB_DISABLED"] = "true"

# ✅ Load cleaned dataset
df = pd.read_excel("/content/medical_qa_cleaned_mixed_vague_half_removed.xlsx")  # Adjust path as needed
print(f"✅ Cleaned dataset size: {len(df)}")
df["formatted"] = df.apply(lambda row: f"### Question:\n{row['Question']}\n\n### Answer:\n{row['Answer']}", axis=1)
class MedicalQADataset(Dataset):
    def __init__(self, data, tokenizer, max_length=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        full_text = self.data.iloc[idx]["formatted"]

        encoding = self.tokenizer(
            full_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        input_ids = encoding["input_ids"].squeeze()
        attention_mask = encoding["attention_mask"].squeeze()

        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

# ✅ Load BioGPT tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("microsoft/biogpt")
model = AutoModelForCausalLM.from_pretrained("microsoft/biogpt")
tokenizer.pad_token = tokenizer.eos_token


dataset = MedicalQADataset(df, tokenizer)
print(f"✅ PyTorch dataset size: {len(dataset)}")


batch_size = 8
train_dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
print(f"✅ Batches per epoch: {len(train_dataloader)}")

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/biogpt_qa_finetuned_ch",
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=500,
    save_steps=1000,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

# ✅ Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

# ✅ Training info
print(f"🔍 Total training steps: {training_args.num_train_epochs * len(train_dataloader)}")
trainer.train()

# ✅ Save Final Model
model.save_pretrained("/content/drive/MyDrive/biogpt_qa_finetuned_ph")
tokenizer.save_pretrained("/content/drive/MyDrive/biogpt_qa_finetuned_ph")
print("🎯 Training complete. Model saved.")


✅ Cleaned dataset size: 24997


config.json:   0%|          | 0.00/595 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

✅ PyTorch dataset size: 24997
✅ Batches per epoch: 3125
🔍 Total training steps: 9375


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
500,1.556300
1000,1.325000
1500,1.240000
2000,1.037000
2500,1.002600
3000,0.992300
3500,0.851800
4000,0.807300
4500,0.803400


🎯 Training complete. Model saved.


In [ ]:
import os
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from torch.utils.data import Dataset

# ✅ Disable wandb logging
os.environ["WANDB_DISABLED"] = "true"

# ✅ Load your cleaned dataset (new entries)
df = pd.read_excel("/content/medicalQAset.xlsx")  # Adjust path if needed
print(f"✅ Dataset size: {len(df)}")

# ✅ Format each row
df["formatted"] = df.apply(lambda row: f"### Question:\n{row['Question']}\n\n### Answer:\n{row['Answer']}", axis=1)

# ✅ Dataset class
class MedicalQADataset(Dataset):
    def __init__(self, data, tokenizer, max_length=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        full_text = self.data.iloc[idx]["formatted"]

        encoding = self.tokenizer(
            full_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        input_ids = encoding["input_ids"].squeeze()
        attention_mask = encoding["attention_mask"].squeeze()

        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

# ✅ Load the tokenizer and model from your previous finetuned model
model_path = "/content/drive/MyDrive/biogpt_qa_refinetuned_final"  # biogpt_qa_finetuned_ph from previous code above
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

tokenizer.pad_token = tokenizer.eos_token

dataset = MedicalQADataset(df, tokenizer)
batch_size = 8
train_dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# ✅ Training arguments for continued finetuning
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/biogpt_qa_refinetuned_final_ch",
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=500,
    save_steps=1000,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

# ✅ Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

# ✅ Training steps info
print(f"🔍 Total training steps: {training_args.num_train_epochs * len(train_dataloader)}")

# ✅ Start training
trainer.train()

# ✅ Save updated model
final_output_dir = "/content/drive/MyDrive/biogpt_qa_refinetuned_final"
model.save_pretrained(final_output_dir)
tokenizer.save_pretrained(final_output_dir)
print("🎯 Re-finetuning complete. Model saved.")


✅ Dataset size: 50050
🔍 Total training steps: 18771


`use_cache=True` is incompatible with gradient checkpointing`. Setting `use_cache=False`...


Step,Training Loss
500,1.184100
1000,1.151200
1500,1.125700
2000,1.108700
2500,1.097900


Step,Training Loss
500,1.184100
1000,1.151200
1500,1.125700
2000,1.108700
2500,1.097900
3000,1.086900
3500,0.912100
4000,0.875200
4500,0.863800
5000,0.866900


🎯 Re-finetuning complete. Model saved.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch import cuda


device = "cuda" if cuda.is_available() else "cpu"
LOAD_PATH = "/content/drive/MyDrive/biogpt_qa_refinetuned_final"

tokenizer = AutoTokenizer.from_pretrained(LOAD_PATH)
model = AutoModelForCausalLM.from_pretrained(LOAD_PATH).to(device)

print("Fine-tuned model loaded! ✅")

Fine-tuned model loaded! ✅


In [ ]:
import torch
def get_answer(question, max_length=200):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    input_text = f"Question: {question}\nAnswer:"
    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=False,
            temperature=1.0,
            num_beams=4,
            no_repeat_ngram_size=2,
            early_stopping=True
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract answer after "Answer:"
    if "Answer:" in decoded:
        answer = decoded.split("Answer:")[-1].strip()
    else:
        answer = decoded.strip()

    # Optional: Trim to first 1–2 sentences
    sentences = answer.split(". ")
    if len(sentences) > 1:
        answer = ". ".join(sentences[:2]) + "."

    return answer



In [ ]:
qa=["What are the symptoms of fever?",

"What are the symptoms of malaria?",

"Can I take paracetamol during pregnancy?",

"How to reduce high blood pressure naturally?"]


for i in qa:
  print(i)
  print(get_answer(i))

What are the symptoms of fever?
Fever is a common sign of infection, raising body temperature to help the immune system fight invaders. Symptoms include fever, chills, body aches, fatigue, and difficulty sleeping.
What are the symptoms of malaria?
malaria causes sudden fever, chills, sweating, and weakness. It spreads through mosquito bites and is treated with artesunate.
Can I take paracetamol during pregnancy?
Yes, in small amounts for safety and effectiveness. Paracetamol helps ease menstrual cramps, supports fetal development, and may reduce the risk of premature birth or low birth weight.
How to reduce high blood pressure naturally?
Limit salt, exercise regularly, eat heart-healthy foods, and manage stress through relaxation techniques. Regular monitoring and lifestyle changes are key.
